In [ ]:
# 标准库与第三方库导入
import json
import logging
import pickle
import random
import tarfile
import urllib.request
from argparse import Namespace
from datetime import datetime
from pathlib import Path

# 可视化、数据处理与深度学习相关库
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import SGD
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.functional as TF
from torchvision.models import ResNet50_Weights
from torchvision.models.segmentation import DeepLabV3_ResNet50_Weights, deeplabv3_resnet50


def resolve_exp_root() -> Path:
    # 定位项目目录，便于 notebook 在不同启动路径下运行
    cwd = Path.cwd().resolve()
    if (cwd / "VOC2012").exists() and (cwd / "downloads").exists():
        return cwd
    if (cwd / "exp2").exists():
        return cwd / "exp2"
    if cwd.name == "exp2":
        return cwd
    for parent in [cwd, *cwd.parents]:
        if (parent / "exp2").exists():
            return parent / "exp2"
    raise RuntimeError("Cannot locate the exp2 directory.")


EXP_ROOT = resolve_exp_root()
EXP_ROOT


In [ ]:
# 数据下载相关配置
DOWNLOAD_CONFIG = {
    "data_root": EXP_ROOT / "VOC2012",
    "downloads_dir": EXP_ROOT / "downloads",
    "url": "https://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar",
    "force_extract": False,
}

# 实验级公共配置：输出目录、实验名、设备等
EXPERIMENT_CONFIG = {
    "data_root": EXP_ROOT / "VOC2012",
    "output_root": EXP_ROOT / "outputs",
    # 设为 None 时，自动使用时间戳作为实验目录名
    "experiment_name": None,
    "device": "auto",
}

# 训练阶段配置
TRAIN_CONFIG = {
    "train_split": "train",
    "val_split": "val",
    "epochs": 3,
    "batch_size": 4,
    "num_workers": 2,
    "crop_size": 513,
    "min_scale": 0.5,
    "max_scale": 2.0,
    "lr": 0.001,
    "momentum": 0.9,
    "weight_decay": 1e-4,
    "num_classes": 21,
    "ignore_label": 255,
    "seed": 42,
    # voc 表示直接使用 torchvision 提供的 VOC 预训练分割权重
    "weights": "voc",
    # 当 weights="none" 时，backbone_weights 才真正生效
    "backbone_weights": "imagenet",
    "resume": None,
    "freeze_bn": False,
    "eval_every": 1,
    "eval_long_size": None,
    "log_interval": 20,
}

# 评估阶段配置
EVALUATE_CONFIG = {
    "split": "val",
    # None 时默认读取当前实验目录下 train/best.pth
    "checkpoint": None,
    "weights": "none",
    "backbone_weights": "imagenet",
    "num_classes": 21,
    "ignore_label": 255,
    "num_workers": 2,
    "long_size": None,
}

# 推理与可视化阶段配置
PREDICT_CONFIG = {
    # input 可以是单张图像路径，也可以是一个目录
    "input": EXP_ROOT / "VOC2012" / "JPEGImages" / "2007_000032.jpg",
    "checkpoint": None,
    "weights": "none",
    "backbone_weights": "imagenet",
    "num_classes": 21,
    "long_size": 513,
}

EXPERIMENT_CONFIG, TRAIN_CONFIG, EVALUATE_CONFIG, PREDICT_CONFIG


In [ ]:
# VOC 21 类调色板，用于把预测类别索引映射回彩色分割图
VOC_COLORMAP = [
    (0, 0, 0), (128, 0, 0), (0, 128, 0), (128, 128, 0), (0, 0, 128),
    (128, 0, 128), (0, 128, 128), (128, 128, 128), (64, 0, 0), (192, 0, 0),
    (64, 128, 0), (192, 128, 0), (64, 0, 128), (192, 0, 128), (64, 128, 128),
    (192, 128, 128), (0, 64, 0), (128, 64, 0), (0, 192, 0), (128, 192, 0),
    (0, 64, 128),
]

# VOC 21 类类别名，对应评估输出中的类别 IoU
VOC_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat",
    "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person", "pottedplant",
    "sheep", "sofa", "train", "tvmonitor",
]

# ImageNet 归一化参数
IMAGE_MEAN = (0.485, 0.456, 0.406)
IMAGE_STD = (0.229, 0.224, 0.225)
VOC2012_URL = "https://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar"


def ensure_dir(path):
    # 保证目录存在，不存在则递归创建
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def build_run_stamp():
    # 当前时间戳既用于日志命名，也用于实验子目录命名
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def get_latest_experiment_marker(output_root):
    return Path(output_root) / "latest_experiment.txt"


def set_latest_experiment(output_root, experiment_dir):
    # 记录最近一次训练的实验目录，便于评估和推理默认跟随
    marker = get_latest_experiment_marker(output_root)
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text(str(Path(experiment_dir).resolve()), encoding="utf-8")


def get_latest_experiment(output_root):
    marker = get_latest_experiment_marker(output_root)
    if not marker.exists():
        raise FileNotFoundError(f"Latest experiment marker not found: {marker}")
    return Path(marker.read_text(encoding="utf-8").strip())


def create_experiment_dir(output_root, experiment_name=None):
    # 新建一个实验目录；如果 experiment_name 为空，则自动用时间戳
    root = ensure_dir(output_root)
    experiment_name = experiment_name or build_run_stamp()
    experiment_dir = ensure_dir(root / experiment_name)
    set_latest_experiment(root, experiment_dir)
    return experiment_dir


def resolve_experiment_dir(output_root, experiment_name=None):
    # 解析已有实验目录；experiment_name="latest" 时跟随最近一次训练
    root = ensure_dir(output_root)
    if experiment_name in (None, "", "latest"):
        experiment_dir = get_latest_experiment(root)
    else:
        experiment_dir = root / experiment_name
    if not experiment_dir.exists():
        raise FileNotFoundError(f"Experiment directory not found: {experiment_dir}")
    return experiment_dir


def setup_logger(name, log_file):
    # 同时把日志写到终端和文件，便于实验记录与回看
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.propagate = False
    for handler in list(logger.handlers):
        logger.removeHandler(handler)
        handler.close()
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(formatter)
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)
    return logger


def make_serializable(data):
    # 将 Path 等不可直接 JSON 序列化的对象转换为基础类型
    if isinstance(data, dict):
        return {k: make_serializable(v) for k, v in data.items()}
    if isinstance(data, list):
        return [make_serializable(v) for v in data]
    if isinstance(data, tuple):
        return [make_serializable(v) for v in data]
    if isinstance(data, Path):
        return str(data)
    return data


def save_json(data, path):
    # 保存实验配置、评估结果、预测清单等结构化信息
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(make_serializable(data), f, indent=2, ensure_ascii=False)


def save_checkpoint(state, path):
    # 保存训练 checkpoint，并将元数据转换为可安全序列化格式
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(make_serializable(state), path)


def load_checkpoint(path, map_location="cpu"):
    path = Path(path)
    try:
        return torch.load(path, map_location=map_location)
    except pickle.UnpicklingError:
        return torch.load(path, map_location=map_location, weights_only=False)


def checkpoint_uses_aux_classifier(checkpoint):
    state_dict = checkpoint.get("model", checkpoint)
    return any(key.startswith("aux_classifier.") for key in state_dict.keys())


def seed_everything(seed):
    # 固定随机种子，尽量保证实验结果可复现
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device(device_name="auto"):
    # 根据配置自动选择训练设备
    if device_name == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(device_name)


def download_with_progress(url, destination):
    # 下载 VOC 压缩包，并在终端显示下载进度
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    def report(blocks, block_size, total_size):
        if total_size <= 0:
            return
        downloaded = min(blocks * block_size, total_size)
        ratio = downloaded / total_size
        print(f"\rDownloading: {ratio:6.2%}", end="")

    urllib.request.urlretrieve(url, destination, report)
    print()
    return destination


def extract_tar(archive_path, target_dir):
    # 解压 VOC tar 文件，并返回 VOC2012 根目录
    target_dir = ensure_dir(target_dir)
    with tarfile.open(archive_path, "r") as tar:
        tar.extractall(target_dir)
    return target_dir / "VOCdevkit" / "VOC2012"


def prepare_voc2012(data_root, downloads_dir, archive_url=VOC2012_URL, force_extract=False):
    # 数据准备总入口：若本地已有数据则直接复用，否则自动下载并解压
    data_root = Path(data_root)
    downloads_dir = Path(downloads_dir)
    if (data_root / "JPEGImages").exists() and not force_extract:
        print(f"Dataset is ready: {data_root}")
        return data_root
    archive_path = downloads_dir / Path(archive_url).name
    if not archive_path.exists():
        print(f"Archive not found locally, downloading from {archive_url}")
        download_with_progress(archive_url, archive_path)
    else:
        print(f"Using local archive: {archive_path}")
    extracted_root = extract_tar(archive_path, data_root.parent)
    if extracted_root != data_root and extracted_root.exists() and not data_root.exists():
        extracted_root.rename(data_root)
    print(f"Dataset prepared at: {data_root}")
    return data_root


class AverageMeter:
    # 统计训练或评估过程中某个指标的平均值
    def __init__(self):
        self.reset()

    def reset(self):
        self.sum = 0.0
        self.count = 0

    def update(self, value, n=1):
        self.sum += value * n
        self.count += n

    @property
    def average(self):
        return self.sum / max(self.count, 1)


In [ ]:
class RandomScaleCropFlip:
    # 训练时的数据增强：随机缩放、随机裁剪、随机翻转
    def __init__(self, crop_size, min_scale, max_scale, ignore_label):
        self.crop_size = crop_size
        self.min_scale = min_scale
        self.max_scale = max_scale
        self.ignore_label = ignore_label

    def __call__(self, image, mask):
        scale = random.uniform(self.min_scale, self.max_scale)
        new_width = max(1, int(image.width * scale))
        new_height = max(1, int(image.height * scale))
        image = image.resize((new_width, new_height), resample=Image.BILINEAR)
        mask = mask.resize((new_width, new_height), resample=Image.NEAREST)

        # 如果缩放后尺寸小于裁剪尺寸，则先补边
        pad_width = max(self.crop_size - new_width, 0)
        pad_height = max(self.crop_size - new_height, 0)
        if pad_width or pad_height:
            image = ImageOps.expand(image, border=(0, 0, pad_width, pad_height), fill=0)
            mask = ImageOps.expand(mask, border=(0, 0, pad_width, pad_height), fill=self.ignore_label)

        left = random.randint(0, image.width - self.crop_size)
        top = random.randint(0, image.height - self.crop_size)
        image = image.crop((left, top, left + self.crop_size, top + self.crop_size))
        mask = mask.crop((left, top, left + self.crop_size, top + self.crop_size))

        if random.random() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)

        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=IMAGE_MEAN, std=IMAGE_STD)
        mask = TF.pil_to_tensor(mask).long().squeeze(0)
        return image, mask


class EvalTransform:
    # 评估阶段的预处理：可选长边缩放 + 归一化
    def __init__(self, long_size=None):
        self.long_size = long_size

    def __call__(self, image, mask):
        image = image.convert("RGB")
        if self.long_size is not None:
            scale = self.long_size / max(image.height, image.width)
            new_width = max(1, int(round(image.width * scale)))
            new_height = max(1, int(round(image.height * scale)))
            image = image.resize((new_width, new_height), resample=Image.BILINEAR)
        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=IMAGE_MEAN, std=IMAGE_STD)
        mask = TF.pil_to_tensor(mask).long().squeeze(0)
        return image, mask


class PredictTransform:
    # 推理阶段预处理：保留原图尺寸，后续再把预测结果插值回原分辨率
    def __init__(self, long_size=None):
        self.long_size = long_size

    def __call__(self, image):
        image = image.convert("RGB")
        original_size = image.size
        if self.long_size is not None:
            scale = self.long_size / max(image.height, image.width)
            new_width = max(1, int(round(image.width * scale)))
            new_height = max(1, int(round(image.height * scale)))
            image = image.resize((new_width, new_height), resample=Image.BILINEAR)
        tensor = TF.to_tensor(image)
        tensor = TF.normalize(tensor, mean=IMAGE_MEAN, std=IMAGE_STD)
        return tensor, original_size


class VOCSegmentationDataset(Dataset):
    # 读取 VOC2012 语义分割数据集
    def __init__(self, data_root, split="train", transform=None):
        self.data_root = Path(data_root)
        self.split = split
        self.transform = transform
        split_file = self.data_root / "ImageSets" / "Segmentation" / f"{split}.txt"
        if not split_file.exists():
            raise FileNotFoundError(f"Split file not found: {split_file}")
        self.sample_ids = [line.strip() for line in split_file.read_text().splitlines() if line.strip()]
        self.images_dir = self.data_root / "JPEGImages"
        self.masks_dir = self.data_root / "SegmentationClass"

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, index):
        sample_id = self.sample_ids[index]
        image = Image.open(self.images_dir / f"{sample_id}.jpg").convert("RGB")
        mask = Image.open(self.masks_dir / f"{sample_id}.png")
        if self.transform is not None:
            image, mask = self.transform(image, mask)
        return image, mask, sample_id


class SegmentationMetric:
    # 计算 pixel accuracy 和 mIoU 所需的混淆矩阵
    def __init__(self, num_classes, ignore_index=255):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.confusion_matrix = torch.zeros((num_classes, num_classes), dtype=torch.float64)

    @torch.no_grad()
    def update(self, prediction, target):
        prediction = prediction.view(-1).cpu()
        target = target.view(-1).cpu()
        valid = target != self.ignore_index
        prediction = prediction[valid]
        target = target[valid]
        if prediction.numel() == 0:
            return
        indices = self.num_classes * target + prediction
        bins = torch.bincount(indices, minlength=self.num_classes ** 2)
        self.confusion_matrix += bins.reshape(self.num_classes, self.num_classes)

    def compute(self):
        hist = self.confusion_matrix
        diagonal = torch.diag(hist)
        union = hist.sum(dim=1) + hist.sum(dim=0) - diagonal
        pixel_acc = (diagonal.sum() / hist.sum()).item() if hist.sum() > 0 else 0.0
        per_class_iou = torch.where(union > 0, diagonal / union, torch.zeros_like(union))
        mean_iou = per_class_iou.mean().item()
        return {
            "pixel_accuracy": pixel_acc,
            "mean_iou": mean_iou,
            "per_class_iou": per_class_iou.tolist(),
        }


In [ ]:
def resolve_segmentation_weights(name):
    # 解析分割模型预训练权重配置
    if name == "none":
        return None
    if name == "voc":
        return DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
    raise ValueError(f"Unsupported segmentation weights: {name}")


def resolve_backbone_weights(name):
    # 解析主干网络 ResNet50 的预训练权重配置
    if name == "none":
        return None
    if name == "imagenet":
        return getattr(ResNet50_Weights, "IMAGENET1K_V2", ResNet50_Weights.IMAGENET1K_V1)
    raise ValueError(f"Unsupported backbone weights: {name}")


def build_deeplabv3_resnet50_model(num_classes=21, weights="none", backbone_weights="imagenet", aux_loss=None):
    # 构建 DeepLabV3-ResNet50，并根据 checkpoint 需要决定是否保留 aux classifier
    segmentation_weights = resolve_segmentation_weights(weights)
    if segmentation_weights is not None and num_classes != 21:
        raise ValueError("VOC pretrained segmentation weights only support num_classes=21.")
    model = deeplabv3_resnet50(
        weights=segmentation_weights,
        weights_backbone=None if segmentation_weights is not None else resolve_backbone_weights(backbone_weights),
        num_classes=num_classes,
        aux_loss=aux_loss,
    )
    return model


def freeze_batch_norm(module):
    # 冻结所有 BatchNorm 层的统计量和参数
    for child in module.modules():
        if isinstance(child, nn.modules.batchnorm._BatchNorm):
            child.eval()
            for parameter in child.parameters():
                parameter.requires_grad = False


def train_one_epoch(model, loader, optimizer, criterion, device, epoch, scheduler=None, scaler=None, log_interval=20, logger=None):
    # 单个 epoch 的训练过程
    model.train()
    loss_meter = AverageMeter()
    amp_enabled = scaler is not None and device.type == "cuda"
    for step, (images, masks, _) in enumerate(loader, start=1):
        images = images.to(device, non_blocking=device.type == "cuda")
        masks = masks.to(device, non_blocking=device.type == "cuda")
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            outputs = model(images)["out"]
            loss = criterion(outputs, masks)
        if amp_enabled:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        if scheduler is not None:
            scheduler.step()
        loss_meter.update(loss.item(), images.size(0))
        if logger is not None and (step % log_interval == 0 or step == len(loader)):
            logger.info(
                "Epoch %02d | Step %04d/%04d | lr=%.6f | loss=%.4f",
                epoch,
                step,
                len(loader),
                optimizer.param_groups[0]["lr"],
                loss_meter.average,
            )
    return {"loss": loss_meter.average}


@torch.no_grad()
def evaluate_model(model, loader, device, num_classes, ignore_index):
    # 验证集评估，输出 loss、pixel accuracy 和 mIoU
    model.eval()
    metric = SegmentationMetric(num_classes=num_classes, ignore_index=ignore_index)
    loss_meter = AverageMeter()
    criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)
    for images, masks, _ in loader:
        images = images.to(device, non_blocking=device.type == "cuda")
        masks = masks.to(device, non_blocking=device.type == "cuda")
        outputs = model(images)["out"]
        if outputs.shape[-2:] != masks.shape[-2:]:
            outputs = F.interpolate(outputs, size=masks.shape[-2:], mode="bilinear", align_corners=False)
        loss = criterion(outputs, masks)
        predictions = outputs.argmax(dim=1)
        metric.update(predictions, masks)
        loss_meter.update(loss.item(), images.size(0))
    scores = metric.compute()
    scores["loss"] = loss_meter.average
    return scores


def colorize_mask(mask):
    # 将类别索引图转换成 VOC 调色板彩色标签图
    palette = []
    for color in VOC_COLORMAP:
        palette.extend(color)
    palette.extend([0] * (768 - len(palette)))
    image = Image.fromarray(mask.astype(np.uint8), mode="P")
    image.putpalette(palette)
    return image


def blend_overlay(image, mask, alpha=0.6):
    # 将预测掩码与原图叠加，生成更易观察的可视化结果
    color_mask = colorize_mask(mask).convert("RGB")
    return Image.blend(image.convert("RGB"), color_mask, alpha=alpha)


def iter_input_images(path):
    # 支持对单张图或目录批量推理
    path = Path(path)
    if path.is_file():
        return [path]
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    return sorted([item for item in path.iterdir() if item.suffix.lower() in exts])


def run_training(args):
    # 训练主流程：创建实验目录、训练模型、记录日志与 checkpoint
    seed_everything(args.seed)
    device = get_device(args.device)
    experiment_dir = create_experiment_dir(args.output_root, args.experiment_name)
    train_dir = ensure_dir(experiment_dir / "train")
    run_stamp = build_run_stamp()
    logger = setup_logger("notebook.segmentation.train", train_dir / "logs" / f"train_{run_stamp}.log")
    logger.info("Training started")
    logger.info("Experiment directory: %s", experiment_dir)
    logger.info("Arguments: %s", vars(args))

    train_dataset = VOCSegmentationDataset(
        data_root=args.data_root,
        split=args.train_split,
        transform=RandomScaleCropFlip(args.crop_size, args.min_scale, args.max_scale, args.ignore_label),
    )
    val_dataset = VOCSegmentationDataset(
        data_root=args.data_root,
        split=args.val_split,
        transform=EvalTransform(args.eval_long_size),
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=device.type == "cuda",
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=device.type == "cuda",
    )
    logger.info("Training samples: %d", len(train_dataset))
    logger.info("Validation samples: %d", len(val_dataset))
    logger.info("Device: %s", device)

    # 如果需要断点续训，则先从 checkpoint 推断模型是否带 aux classifier
    resume_checkpoint = None
    aux_loss = None
    if args.resume is not None:
        resume_checkpoint = load_checkpoint(args.resume, map_location="cpu")
        aux_loss = checkpoint_uses_aux_classifier(resume_checkpoint)

    model = build_deeplabv3_resnet50_model(
        num_classes=args.num_classes,
        weights=args.weights,
        backbone_weights=args.backbone_weights,
        aux_loss=aux_loss,
    ).to(device)
    if args.freeze_bn:
        freeze_batch_norm(model)

    criterion = nn.CrossEntropyLoss(ignore_index=args.ignore_label)
    optimizer = SGD(
        params=(parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=args.lr,
        momentum=args.momentum,
        weight_decay=args.weight_decay,
    )
    total_steps = max(len(train_loader) * args.epochs, 1)
    scheduler = LambdaLR(optimizer, lr_lambda=lambda step: max((1 - step / total_steps) ** 0.9, 0.0))
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

    start_epoch = 1
    best_miou = -1.0
    history = []
    if resume_checkpoint is not None:
        model.load_state_dict(resume_checkpoint["model"])
        optimizer.load_state_dict(resume_checkpoint["optimizer"])
        scheduler.load_state_dict(resume_checkpoint["scheduler"])
        start_epoch = resume_checkpoint["epoch"] + 1
        best_miou = resume_checkpoint.get("best_miou", best_miou)

    # 保存本次训练配置和实验元信息，便于复现实验
    save_json(
        {
            "run_stamp": run_stamp,
            "experiment_dir": experiment_dir,
            "train_dir": train_dir,
            "args": vars(args),
            "device": str(device),
        },
        train_dir / "logs" / f"train_config_{run_stamp}.json",
    )
    save_json(
        {
            "experiment_dir": experiment_dir,
            "train_dir": train_dir,
            "latest_train_run_stamp": run_stamp,
        },
        experiment_dir / "experiment_info.json",
    )

    for epoch in range(start_epoch, args.epochs + 1):
        train_stats = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            epoch=epoch,
            scheduler=scheduler,
            scaler=scaler,
            log_interval=args.log_interval,
            logger=logger,
        )
        logger.info("Epoch %02d training loss: %.4f", epoch, train_stats["loss"])

        if epoch % args.eval_every != 0:
            history.append({"epoch": epoch, "train_loss": train_stats["loss"], "evaluated": False, "best_miou": best_miou})
            save_json(history, train_dir / "logs" / f"train_history_{run_stamp}.json")
            continue

        # 每隔若干 epoch 在验证集上评估一次，并据此更新最佳模型
        val_stats = evaluate_model(model, val_loader, device, args.num_classes, args.ignore_label)
        logger.info(
            "Epoch %02d validation | loss=%.4f | pixel_acc=%.4f | mIoU=%.4f",
            epoch,
            val_stats["loss"],
            val_stats["pixel_accuracy"],
            val_stats["mean_iou"],
        )

        checkpoint = {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_miou": best_miou,
            "args": vars(args),
        }
        save_checkpoint(checkpoint, train_dir / "last.pth")
        if val_stats["mean_iou"] > best_miou:
            best_miou = float(val_stats["mean_iou"])
            checkpoint["best_miou"] = best_miou
            save_checkpoint(checkpoint, train_dir / "best.pth")

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_stats["loss"],
                "evaluated": True,
                "val_loss": val_stats["loss"],
                "pixel_accuracy": val_stats["pixel_accuracy"],
                "mean_iou": val_stats["mean_iou"],
                "best_miou": best_miou,
            }
        )
        save_json(history, train_dir / "logs" / f"train_history_{run_stamp}.json")

    save_json(
        {
            "run_stamp": run_stamp,
            "experiment_dir": experiment_dir,
            "train_dir": train_dir,
            "best_miou": best_miou,
            "epochs_completed": args.epochs,
            "history_file": str(train_dir / "logs" / f"train_history_{run_stamp}.json"),
        },
        train_dir / "logs" / f"train_summary_{run_stamp}.json",
    )
    logger.info("Training finished. Best mIoU: %.4f", best_miou)
    return experiment_dir


def run_evaluation(args):
    # 评估主流程：加载训练好的模型，并在验证集上计算指标
    device = get_device(args.device)
    experiment_dir = resolve_experiment_dir(args.output_root, args.experiment_name)
    run_stamp = build_run_stamp()
    evaluate_dir = ensure_dir(experiment_dir / "evaluate" / run_stamp)
    logger = setup_logger("notebook.segmentation.evaluate", evaluate_dir / f"evaluate_{run_stamp}.log")
    logger.info("Evaluation started")
    logger.info("Experiment directory: %s", experiment_dir)
    logger.info("Arguments: %s", vars(args))

    dataset = VOCSegmentationDataset(args.data_root, split=args.split, transform=EvalTransform(args.long_size))
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=args.num_workers, pin_memory=device.type == "cuda")

    checkpoint_path = args.checkpoint
    if checkpoint_path is None and args.weights == "none":
        checkpoint_path = experiment_dir / "train" / "best.pth"
    checkpoint = None
    aux_loss = None
    if checkpoint_path is not None:
        checkpoint = load_checkpoint(checkpoint_path, map_location="cpu")
        aux_loss = checkpoint_uses_aux_classifier(checkpoint)

    model = build_deeplabv3_resnet50_model(
        num_classes=args.num_classes,
        weights=args.weights,
        backbone_weights=args.backbone_weights,
        aux_loss=aux_loss,
    ).to(device)
    if checkpoint is not None:
        state_dict = checkpoint["model"] if "model" in checkpoint else checkpoint
        model.load_state_dict(state_dict)
        logger.info("Loaded checkpoint: %s", checkpoint_path)

    metrics = evaluate_model(model, loader, device, args.num_classes, args.ignore_label)
    payload = {
        "run_stamp": run_stamp,
        "experiment_dir": experiment_dir,
        "evaluate_dir": evaluate_dir,
        "checkpoint": checkpoint_path,
        "weights": args.weights,
        "backbone_weights": args.backbone_weights,
        "split": args.split,
        "device": str(device),
        "metrics": metrics,
    }
    save_json(payload, evaluate_dir / f"evaluation_metrics_{run_stamp}.json")
    logger.info("loss: %.4f", metrics["loss"])
    logger.info("pixel_accuracy: %.4f", metrics["pixel_accuracy"])
    logger.info("mean_iou: %.4f", metrics["mean_iou"])
    return metrics, evaluate_dir


def run_prediction(args):
    # 推理主流程：生成彩色分割结果与叠加可视化图
    device = get_device(args.device)
    experiment_dir = resolve_experiment_dir(args.output_root, args.experiment_name)
    run_stamp = build_run_stamp()
    predict_dir = ensure_dir(experiment_dir / "predict" / run_stamp)
    logger = setup_logger("notebook.segmentation.predict", predict_dir / f"predict_{run_stamp}.log")
    logger.info("Prediction started")
    logger.info("Experiment directory: %s", experiment_dir)
    logger.info("Arguments: %s", vars(args))

    checkpoint_path = args.checkpoint
    if checkpoint_path is None and args.weights == "none":
        checkpoint_path = experiment_dir / "train" / "best.pth"
    checkpoint = None
    aux_loss = None
    if checkpoint_path is not None:
        checkpoint = load_checkpoint(checkpoint_path, map_location="cpu")
        aux_loss = checkpoint_uses_aux_classifier(checkpoint)

    model = build_deeplabv3_resnet50_model(
        num_classes=args.num_classes,
        weights=args.weights,
        backbone_weights=args.backbone_weights,
        aux_loss=aux_loss,
    ).to(device)
    if checkpoint is not None:
        state_dict = checkpoint["model"] if "model" in checkpoint else checkpoint
        model.load_state_dict(state_dict)
        logger.info("Loaded checkpoint: %s", checkpoint_path)
    model.eval()

    transform = PredictTransform(args.long_size)
    records = []
    for image_path in iter_input_images(args.input):
        image = Image.open(image_path).convert("RGB")
        tensor, original_size = transform(image)
        tensor = tensor.unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(tensor)["out"]
            output = F.interpolate(output, size=(original_size[1], original_size[0]), mode="bilinear", align_corners=False)
            prediction = output.argmax(dim=1).squeeze(0).cpu().numpy()
        mask = colorize_mask(prediction)
        overlay = blend_overlay(image, prediction)
        mask_path = predict_dir / f"{image_path.stem}_mask.png"
        overlay_path = predict_dir / f"{image_path.stem}_overlay.png"
        mask.save(mask_path)
        overlay.save(overlay_path)
        records.append(
            {
                "input_image": image_path,
                "mask_path": mask_path,
                "overlay_path": overlay_path,
                "original_size": {"width": original_size[0], "height": original_size[1]},
            }
        )
        logger.info("Saved prediction outputs for %s", image_path.name)

    manifest_path = predict_dir / f"prediction_manifest_{run_stamp}.json"
    save_json(
        {
            "run_stamp": run_stamp,
            "experiment_dir": experiment_dir,
            "predict_dir": predict_dir,
            "checkpoint": checkpoint_path,
            "weights": args.weights,
            "backbone_weights": args.backbone_weights,
            "device": str(device),
            "items": records,
        },
        manifest_path,
    )
    logger.info("Saved prediction manifest: %s", manifest_path)
    return predict_dir


def run_full_experiment():
    # 串联完整实验流程：训练 -> 评估 -> 推理
    shared = {
        "data_root": EXPERIMENT_CONFIG["data_root"],
        "output_root": EXPERIMENT_CONFIG["output_root"],
        "experiment_name": EXPERIMENT_CONFIG["experiment_name"],
        "device": EXPERIMENT_CONFIG["device"],
    }

    train_args = Namespace(**shared, **TRAIN_CONFIG)
    run_training(train_args)

    experiment_name = EXPERIMENT_CONFIG["experiment_name"] or "latest"
    experiment_dir = resolve_experiment_dir(EXPERIMENT_CONFIG["output_root"], experiment_name)
    actual_experiment_name = experiment_dir.name

    pipeline_stamp = build_run_stamp()
    pipeline_log = experiment_dir / f"experiment_{pipeline_stamp}.log"
    logger = setup_logger("notebook.segmentation.pipeline", pipeline_log)
    logger.info("Full experiment started")
    logger.info("Experiment directory: %s", experiment_dir)
    logger.info("Experiment config: %s", EXPERIMENT_CONFIG)

    evaluate_args = Namespace(
        **{
            "data_root": EXPERIMENT_CONFIG["data_root"],
            "output_root": EXPERIMENT_CONFIG["output_root"],
            "experiment_name": actual_experiment_name,
            "device": EXPERIMENT_CONFIG["device"],
        },
        **EVALUATE_CONFIG,
    )
    metrics, evaluate_dir = run_evaluation(evaluate_args)
    logger.info("Evaluation finished | mIoU=%.4f", metrics["mean_iou"])

    predict_args = Namespace(
        **{
            "output_root": EXPERIMENT_CONFIG["output_root"],
            "experiment_name": actual_experiment_name,
            "device": EXPERIMENT_CONFIG["device"],
        },
        **PREDICT_CONFIG,
    )
    predict_dir = run_prediction(predict_args)
    logger.info("Prediction finished")

    summary = {
        "pipeline_stamp": pipeline_stamp,
        "experiment_dir": experiment_dir,
        "actual_experiment_name": actual_experiment_name,
        "experiment_config": EXPERIMENT_CONFIG,
        "train_config": TRAIN_CONFIG,
        "evaluate_config": EVALUATE_CONFIG,
        "predict_config": PREDICT_CONFIG,
        "evaluation_metrics": metrics,
        "evaluate_dir": evaluate_dir,
        "predict_dir": predict_dir,
    }
    summary_path = experiment_dir / f"experiment_summary_{pipeline_stamp}.json"
    save_json(summary, summary_path)
    logger.info("Saved experiment summary: %s", summary_path)
    logger.info("Full experiment finished")
    return experiment_dir, metrics, summary_path


In [ ]:
# 如果本地已经存在 VOC2012 数据集，可以跳过本单元。
# 需要重新准备数据时，去掉下面这行前面的注释符号。
# prepare_voc2012(**DOWNLOAD_CONFIG)


In [ ]:
# 执行完整实验：训练、评估、推理会按顺序全部运行
experiment_dir, metrics, summary_path = run_full_experiment()
print("experiment_dir:", experiment_dir)
print("summary_path:", summary_path)
print("evaluation_metrics:", metrics)


In [ ]:
# 遍历当前实验目录，查看 train / evaluate / predict 三部分输出文件
for path in sorted(experiment_dir.rglob("*")):
    rel_path = path.relative_to(experiment_dir)
    prefix = "[DIR]" if path.is_dir() else "[FILE]"
    print(prefix, rel_path)


In [ ]:
# 读取最近一次推理结果中的 overlay 图，便于在 notebook 中直接查看可视化效果
predict_dirs = sorted((experiment_dir / "predict").glob("*")) if (experiment_dir / "predict").exists() else []
if predict_dirs:
    latest_predict_dir = predict_dirs[-1]
    overlay_images = sorted(latest_predict_dir.glob("*_overlay.png"))
    if overlay_images:
        image = Image.open(overlay_images[0]).convert("RGB")
        plt.figure(figsize=(8, 6))
        plt.imshow(image)
        plt.title(overlay_images[0].name)
        plt.axis("off")
        plt.show()
    else:
        print("No overlay image found in the latest predict directory.")
else:
    print("No predict directory found for this experiment.")
